# Sparse Autoencoders (SAEs) & Mechanistic Interpretability: Disentangling Neural Superposition

> **Topic**: AI Interpretability, Monosemantic Dictionary Learning, Top-K SAEs  
> **Key Breakthrough**: Decomposing polysemantic transformer neurons into thousands of human-interpretable atomic features.

---

## 1. The Superposition Hypothesis
Neural networks represent far more real-world features than they have hidden dimensions ($N \gg D$). To fit $N$ concepts into $D$ dimensions, the model stores features in **almost-orthogonal polysemantic superposition**. As a result, a single neuron fires for completely unrelated concepts (e.g. Python code, Golden Gate Bridge, and French literature).

### Sparse Dictionary Learning
A Sparse Autoencoder (SAE) projects the dense hidden state $\mathbf{x} \in \mathbb{R}^D$ into an overcomplete sparse latent dictionary $\mathbf{f} \in \mathbb{R}^M$ ($M \gg D$, e.g., $16\times$ or $32\times$ expansion):
$$\mathbf{f} = \text{ReLU}(W_{\text{enc}} (\mathbf{x} - \mathbf{b}_{\text{dec}}) + \mathbf{b}_{\text{enc}})$$
$$\hat{\mathbf{x}} = W_{\text{dec}} \mathbf{f} + \mathbf{b}_{\text{dec}}$$

### Top-K Sparsity (Gao et al., OpenAI 2024)
Instead of tuning fragile $L_1$ penalty coefficients $\lambda \|\mathbf{f}\|_1$, Top-K SAEs directly keep only the $K$ largest latent activations and set the rest to zero:
$$\mathbf{f}_{\text{Top-}K} = \text{TopK}(\mathbf{f}, K)$$
This completely eliminates shrinkage bias and guarantees exact computation budgets during interpretability auditing!


In [ ]:
import torch
import torch.nn as nn

class TopKSparseAutoencoder(nn.Module):
    def __init__(self, d_model=768, expansion=16, k=32):
        super().__init__()
        self.d_latent = d_model * expansion
        self.k = k
        self.encoder = nn.Linear(d_model, self.d_latent)
        self.decoder = nn.Linear(self.d_latent, d_model, bias=False)
        self.bias_dec = nn.Parameter(torch.zeros(d_model))
        
    def forward(self, x):
        # Center input
        x_cent = x - self.bias_dec
        # Encode
        latents = F.relu(self.encoder(x_cent))
        # Top-K Sparsity Mask
        topk_vals, topk_idx = torch.topk(latents, self.k, dim=-1)
        sparse_latents = torch.zeros_like(latents).scatter_(-1, topk_idx, topk_vals)
        # Decode
        x_reconstructed = self.decoder(sparse_latents) + self.bias_dec
        return x_reconstructed, sparse_latents

sae = TopKSparseAutoencoder(d_model=512, expansion=8, k=16)
x = torch.randn(4, 512)
x_rec, latents = sae(x)
print(f"Dense Input: {x.shape} -> Latent Dictionary: {latents.shape} (Active: {sae.k}/{sae.d_latent})")
print(f"Reconstruction Loss: {F.mse_loss(x_rec, x).item():.4f}")
